# Optimizer profile: parameter fraction and micro-batching
The two knobs that shrink the Jacobian: fewer columns (`param_fraction`) or fewer rows (`microbatch_size`).

In [ ]:
import sys
sys.path.insert(0, '.')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from style import set_style
import profile_helpers as ph
set_style()
PLOT_DIR = 'plots_v2/profile'
df = ph.load_profiles()          # ../profile_results_v2/*/*.json -> one tidy row per configuration
ARCHS = [a for a in ph.ARCH_ORDER if a in set(df.arch)]
df.groupby(['arch', 'study']).size().unstack(fill_value=0)

### Parameter fraction
One line per Sven variant and mask structure (`elementwise`, `rows`, `tensor`). Dotted: the ideal cost proportional to the fraction, anchored at the unmasked point. Hooks-capture masking is not implemented for transformers, so nanoGPT shows the `jacrev` variants only.

In [ ]:
for value, fname in [('step_ms', 'paramfrac_step_time'), ('peak_mb', 'paramfrac_peak_memory')]:
    fig, axes = plt.subplots(1, len(ARCHS), figsize=(5.2 * len(ARCHS), 4.6), squeeze=False)
    for ax, arch in zip(axes[0], ARCHS):
        ph.plot_sweep(df, arch, 'param_fraction', 'pf', ax, value=value, style_by='mask_mode', ideal='linear')
    ph.legend_below(fig, axes[0][0])
    fig.tight_layout(); ph.save(fig, f'{fname}.pdf', PLOT_DIR); plt.show()

In [ ]:
pf = df[df.study == 'param_fraction']
pf.pivot_table(index=['arch', 'method', 'mask_mode'], columns='pf', values='step_ms').round(2)

In [ ]:
pf.pivot_table(index=['arch', 'method', 'mask_mode'], columns='pf', values='peak_mb').round(1)

### Micro-batch size
Rows shrink from $B$ to $B/\mathrm{mb}$ for the `jacrev` variants. Hooks capture always builds the $B\times B$ per-sample kernel and pools it afterwards, so its cost should be flat.

In [ ]:
for value, fname in [('step_ms', 'microbatch_step_time'), ('peak_mb', 'microbatch_peak_memory')]:
    fig, axes = plt.subplots(1, len(ARCHS), figsize=(5 * len(ARCHS), 4.2), squeeze=False)
    for ax, arch in zip(axes[0], ARCHS):
        ph.plot_sweep(df, arch, 'microbatch', 'mb', ax, value=value)
    ph.legend_below(fig, axes[0][0])
    fig.tight_layout(); ph.save(fig, f'{fname}.pdf', PLOT_DIR); plt.show()

In [ ]:
df[df.study == 'microbatch'].pivot_table(index=['arch', 'method'], columns='mb', values='step_ms').round(2)

### Combinations that are not supported

In [ ]:
ph.status_report(df[df.study.isin(['param_fraction', 'microbatch'])])